# Agent memory with Neo4j and Haystack

`Neo4jMemoryRetriever` and `Neo4jMemoryWriter` give any chat generator three kinds of memory, stored in Neo4j by
[`neo4j-agent-memory`](https://neo4j.com/labs/agent-memory/):

- **Short-term**: the session's conversation.
- **Long-term**: preferences, entities and facts that outlive sessions.
- **Reasoning**: traces of past tasks and the tools they used.

You need a Neo4j 5 database (for example `docker run -d -p 7687:7687 -e NEO4J_AUTH=neo4j/testpassword neo4j:5-community`)
and an OpenAI API key, used for entity extraction, embeddings and the chat model.

In [1]:
# From a clone of the repo (or: pip install "haystack-neo4j[memory] @ git+https://github.com/neo4j-labs/neo4j-haystack")
%pip install -q -e "..[memory]" python-dotenv


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Credentials

Neo4j settings default to the local Docker database above. The OpenAI key is read from `.env` (or asked for with
`getpass`) and never printed.

In [2]:
import os
from getpass import getpass

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
os.environ.setdefault("NEO4J_URI", "neo4j+s://c9ae0431.databases.neo4j.io")
os.environ.setdefault("NEO4J_DATABASE", "c9ae0431")
os.environ.setdefault("NEO4J_USERNAME", "c9ae0431")
os.environ.setdefault("NEO4J_PASSWORD", "fn4JGK7ZVsVdMqkHAItHY44OCzPH94W16I5j-2u18o8")
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

OPENAI_API_KEY set: True


## 1. A chat pipeline with all three memories

The retriever reads memory before the LLM; the writer stores the turn after it. Both share one store. The writer
uses the `review` policy for the user's message: the extractor proposes entities, and only people and
organizations above a confidence threshold are kept.

In [ ]:
from haystack import Pipeline
from haystack.components.generators.chat import OpenAIChatGenerator

from haystack_integrations.components.retrievers.neo4j import Neo4jMemoryRetriever
from haystack_integrations.components.writers.neo4j import Neo4jMemoryWriter
from haystack_integrations.memory_stores.neo4j import ExtractionPolicy, Neo4jMemoryStore

store = Neo4jMemoryStore()
policy = ExtractionPolicy(user="review", allowed_types=["PERSON", "ORGANIZATION"], min_confidence=0.6)

pipe = Pipeline()
pipe.add_component("reader", Neo4jMemoryRetriever(store, system_prompt="You are a concise research assistant."))
pipe.add_component("llm", OpenAIChatGenerator(model="gpt-5.4-mini"))
pipe.add_component("writer", Neo4jMemoryWriter(store, extraction_policy=policy))
pipe.connect("reader.messages", "llm.messages")
pipe.connect("llm.replies", "writer.replies")


def chat(query, session_id, **writer_inputs):
    result = pipe.run(
        {
            "reader": {"query": query, "session_id": session_id},
            "writer": {"query": query, "session_id": session_id, **writer_inputs},
        },
        include_outputs_from={"reader","llm"},
    )
    print("Context the model saw:\n", result["reader"]["context"] or "(none)")
    print("\nAnswer:", result["llm"]["replies"][0].text)
    print("\nWritten:", {k: result["writer"][k] for k in ("messages_written", "trace_id")})
    print("Entities linked:", result["writer"]["entities"])
    return result

## 2. Turn 1: a preference and a fact

Preferences and facts can be passed to the writer explicitly. `review` mode can also pick up preferences the
extractor finds in the message.

In [4]:
first = chat(
    "I'm Pravesh from Northwind Capital. I focus on German AI companies. Which ones should I look at?",
    "session-1",
    preferences=[{"category": "region", "preference": "Focus on companies in Germany"}],
    facts=[{"subject": "Pravesh", "predicate": "works_at", "object": "Northwind Capital"}],
)

Context the model saw:
 ## Memory
User preferences:
- [technology] focuses on German AI companies
- [region] Focus on companies in Germany
Known entities:
- German AI companies (ORGANIZATION)
- Northwind Capital (ORGANIZATION)
- Pravesh (PERSON)
Facts:
- Pravesh works_at Northwind Capital
Similar past tasks that succeeded:
- I'm Pravesh from Northwind Capital. I focus on German AI companies. Which ones should I look at? -> If you’re focused on German AI, I’d look at a mix of foundational model players, enterprise software, and applied AI startups. A practical shortlist:…


KeyError: 'llm'

## 3. Turn 2, same session: short-term memory resolves the reference

In [ ]:
second = chat("Which of those has the largest team?", "session-1")
print("\nHistory replayed:", [m.text[:60] for m in second["reader"]["history"]])

## 4. Turn 3, new session: long-term memory brings the preference back

No history here, but the preference and the earlier task are recalled by similarity.

In [ ]:
third = chat("Suggest a few companies for my next deep dive.", "session-2")
print("\nRecalled preferences:", [p["preference"] for p in third["reader"]["memories"]["preferences"]])
print("Recalled traces:", [t["task"] for t in third["reader"]["memories"]["traces"]])

## 5. Tools and reasoning memory

Haystack 3 runs tools inside an `Agent`. Use the agent as a plain pipeline component and send its full transcript
to the writer. The writer keeps only this turn (not the injected history), records a trace with one step per tool
call, and with `assistant="tool_results"` links the companies the tool returned.

In [ ]:
import json

from haystack.components.agents import Agent
from haystack.tools import tool


@tool
def search_companies(country: str, sector: str) -> str:
    """Search a (toy) company database by country and sector. Returns JSON."""
    data = [
        {"name": "Aleph Alpha", "type": "ORGANIZATION", "country": "DE", "sector": "AI"},
        {"name": "DeepL", "type": "ORGANIZATION", "country": "DE", "sector": "AI"},
    ]
    return json.dumps([c for c in data if c["country"] == country.upper()[:2] or country.lower() == "germany"])


agent_pipe = Pipeline()
agent_pipe.add_component("reader", Neo4jMemoryRetriever(store))
agent_pipe.add_component("agent", Agent(chat_generator=OpenAIChatGenerator(model="gpt-5.4-mini"), tools=[search_companies]))
agent_pipe.add_component("writer", Neo4jMemoryWriter(store, extraction_policy=ExtractionPolicy(user="skip", assistant="tool_results")))
agent_pipe.connect("reader.messages", "agent.messages")
agent_pipe.connect("agent.messages", "writer.messages")

task = "Find German AI companies using the company search tool."
out = agent_pipe.run(
    {"reader": {"query": task, "session_id": "session-3"}, "writer": {"query": task, "session_id": "session-3"}}
)
print("Trace:", out["writer"]["trace_id"])
print("Entities from tool results:", out["writer"]["entities"])

In [ ]:
# A similar task later: the reader returns the earlier trace and the tool it used.
recall = Neo4jMemoryRetriever(store).run(query="Which German AI companies exist?", session_id="session-4")
print(recall["context"])

## 6. Look at the graph

In [ ]:
rows = store.run(store.client.query.cypher(
    """
    MATCH (t:ReasoningTrace)-[:HAS_STEP]->(s)-[:USES_TOOL]->(c:ToolCall)-[:INSTANCE_OF]->(tool:Tool)
    RETURN t.task AS task, t.success AS success, tool.name AS tool, c.status AS status
    ORDER BY t.started_at DESC LIMIT 5
    """
))
for row in rows:
    print(row)

## 7. Async and serialization

The same pipeline runs with `run_async`. Saved YAML contains env-var names, never passwords or keys, and a loaded
pipeline's reader and writer share one store.

In [ ]:
result = await pipe.run_async(
    {
        "reader": {"query": "Remind me what region I focus on.", "session_id": "session-5"},
        "writer": {"query": "Remind me what region I focus on.", "session_id": "session-5"},
    },
    include_outputs_from={"llm"},
)
print(result["llm"]["replies"][0].text)

yaml_text = pipe.dumps()
print("Neo4j password in YAML:", os.environ["NEO4J_PASSWORD"] in yaml_text)
print("OpenAI key in YAML:", os.environ["OPENAI_API_KEY"] in yaml_text)
loaded = Pipeline.loads(yaml_text)
print("Shared store after loading:", loaded.get_component("reader").store is loaded.get_component("writer").store)

## 8. Clean up

`delete_session` removes a conversation. Long-term memory extracted from it is kept by design.

In [ ]:
for session in ["session-1", "session-2", "session-3", "session-4", "session-5"]:
    store.delete_session(session)
store.close()
loaded.get_component("reader").store.close()